# Module 00 — Setting Up & Descriptive Statistics Refresher

**Course**: Intermediate Statistics for Econometrics  
**Author**: Andrea Sarcina — University of Padua  
**Website**: [cinosar.github.io](https://cinosar.github.io)

---

## What this module covers

This is the **gateway module**. We have two goals:

1. **Set up your Python environment** for statistical computing — pandas, matplotlib, seaborn, and later scipy/statsmodels
2. **Refresh descriptive statistics** using a real dataset — not by computing things by hand, but by building intuition for what the numbers mean and when they lie to you

### Prerequisites

- Basic Python knowledge (variables, functions, loops, list comprehensions)
- No prior statistics knowledge assumed, but we move at an intermediate pace

### What you'll need

```bash
pip install pandas numpy matplotlib seaborn scipy statsmodels
```

---

## 0.1 — Imports and Setup

Every notebook in this course will start with this block. Run it once and forget about it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting defaults — clean, publication-ready style
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

# Pandas display options
pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 2)

print("Environment ready.")
print(f"pandas {pd.__version__}, numpy {np.__version__}")

---

## 0.2 — Loading Data: The World Bank Dataset

Rather than fabricating toy data, we'll work with **real macroeconomic data** — GDP per capita, life expectancy, and population across countries. This is the kind of data you'll encounter in econometric research.

We'll build a synthetic but realistic dataset based on World Bank indicators. In future modules, we'll connect to real APIs.

In [ ]:
# We create a realistic cross-sectional dataset of countries
# In a real project, you'd fetch this from the World Bank API or a CSV

np.random.seed(42)  # Reproducibility!

n_countries = 120

# Simulate GDP per capita (log-normal distribution — realistic for income data)
log_gdp = np.random.normal(loc=8.5, scale=1.5, size=n_countries)
gdp_per_capita = np.exp(log_gdp)

# Life expectancy: correlated with GDP (richer countries tend to live longer)
life_exp = 50 + 8 * np.log(gdp_per_capita / 1000) + np.random.normal(0, 4, n_countries)
life_exp = np.clip(life_exp, 45, 88)

# Population (also log-normal, highly skewed)
population = np.exp(np.random.normal(loc=16, scale=2, size=n_countries)).astype(int)

# Gini coefficient (inequality measure, 0-100)
gini = 25 + 10 * np.random.randn(n_countries) + 5 * (10 - np.log(gdp_per_capita))
gini = np.clip(gini, 20, 65)

# Region labels
regions = np.random.choice(
    ["Europe", "Asia", "Africa", "Americas", "Oceania"],
    size=n_countries,
    p=[0.25, 0.30, 0.20, 0.20, 0.05]
)

# Build the DataFrame
df = pd.DataFrame({
    "gdp_per_capita": np.round(gdp_per_capita, 2),
    "life_expectancy": np.round(life_exp, 1),
    "population": population,
    "gini_index": np.round(gini, 1),
    "region": regions
})

print(f"Dataset: {df.shape[0]} countries, {df.shape[1]} variables")
df.head(10)

### 🔑 Key pandas concept: `DataFrame.head()`, `shape`, and `dtypes`

A `DataFrame` is essentially a table — rows are observations (countries), columns are variables. Three things you should always check first:

- **`.shape`** — how many rows and columns?
- **`.dtypes`** — what type is each column? (numeric, categorical, text?)
- **`.head(n)`** — what do the first few rows look like?

In [ ]:
# Always inspect your data before doing anything else
print("Shape:", df.shape)
print("\nColumn types:")
print(df.dtypes)
print("\nBasic info:")
df.info()

---

## 0.3 — Measures of Central Tendency

You already know what mean, median, and mode are. The interesting question is: **when does each one tell you something useful, and when does it mislead you?**

### The Mean–Median Gap as a Diagnostic Tool

When mean ≈ median, the distribution is roughly symmetric. When they diverge, something interesting is happening — usually skewness. For a researcher, that divergence is a **signal**, not just a number.

In [ ]:
# Let's compare mean and median for each variable
summary = pd.DataFrame({
    "mean": df[["gdp_per_capita", "life_expectancy", "gini_index"]].mean(),
    "median": df[["gdp_per_capita", "life_expectancy", "gini_index"]].median(),
})
summary["mean_median_gap"] = summary["mean"] - summary["median"]
summary["gap_pct"] = (summary["mean_median_gap"] / summary["median"] * 100).round(1)

print(summary.to_string())
print()
print("Interpretation:")
print(f"  GDP per capita: mean is {summary.loc['gdp_per_capita', 'gap_pct']:.1f}% higher than median → right-skewed")
print(f"  Life expectancy: gap is {summary.loc['life_expectancy', 'gap_pct']:.1f}% → roughly symmetric")
print(f"  Gini index: gap is {summary.loc['gini_index', 'gap_pct']:.1f}%")

### 💡 Why this matters in econometrics

When you see a variable like GDP per capita where the mean is substantially higher than the median, it tells you the distribution is **right-skewed**. This has practical consequences:

1. **OLS regression assumes normally distributed errors** — skewed dependent variables can violate this
2. **The standard fix**: use `log(GDP per capita)` instead — the log transformation pulls in the right tail and makes the distribution more symmetric
3. **Interpretation changes**: coefficients in a log-linear model represent *percentage* changes, not absolute ones

Let's visualise this.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# GDP per capita — skewed
axes[0].hist(df["gdp_per_capita"], bins=25, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(df["gdp_per_capita"].mean(), color="red", linestyle="--", linewidth=2, label=f'Mean: {df["gdp_per_capita"].mean():,.0f}')
axes[0].axvline(df["gdp_per_capita"].median(), color="orange", linestyle="-", linewidth=2, label=f'Median: {df["gdp_per_capita"].median():,.0f}')
axes[0].set_title("GDP per Capita (USD)")
axes[0].legend(fontsize=9)

# Log GDP — much more symmetric
axes[1].hist(np.log(df["gdp_per_capita"]), bins=25, color="seagreen", edgecolor="white", alpha=0.8)
axes[1].axvline(np.log(df["gdp_per_capita"]).mean(), color="red", linestyle="--", linewidth=2, label=f'Mean: {np.log(df["gdp_per_capita"]).mean():.2f}')
axes[1].axvline(np.log(df["gdp_per_capita"]).median(), color="orange", linestyle="-", linewidth=2, label=f'Median: {np.log(df["gdp_per_capita"]).median():.2f}')
axes[1].set_title("log(GDP per Capita)")
axes[1].legend(fontsize=9)

# Life expectancy — roughly symmetric
axes[2].hist(df["life_expectancy"], bins=25, color="coral", edgecolor="white", alpha=0.8)
axes[2].axvline(df["life_expectancy"].mean(), color="red", linestyle="--", linewidth=2, label=f'Mean: {df["life_expectancy"].mean():.1f}')
axes[2].axvline(df["life_expectancy"].median(), color="orange", linestyle="-", linewidth=2, label=f'Median: {df["life_expectancy"].median():.1f}')
axes[2].set_title("Life Expectancy (years)")
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.suptitle("Mean vs. Median: Detecting Skewness", fontsize=14, fontweight="bold", y=1.03)
plt.savefig("fig_00_01_mean_vs_median.png", dpi=150, bbox_inches="tight")
plt.show()
print("Notice how the log transformation makes mean ≈ median — the skewness disappears.")

---

## 0.4 — Measures of Variability

Variability is *at least* as important as central tendency. Two datasets can have the same mean but tell completely different stories.

### Standard Deviation: What It Actually Means

SD is the **typical distance** of observations from the mean. Not the maximum distance, not a boundary — a *typical* distance. For a normal distribution, about 68% of data falls within ±1 SD of the mean.

### Coefficient of Variation: Comparing Apples and Oranges

SD is in the same units as your variable, which makes it useless for comparing variability *across* variables with different scales. GDP per capita is measured in thousands of dollars; life expectancy in years. The **Coefficient of Variation (CV)** — SD divided by the mean, expressed as a percentage — solves this problem.

In [ ]:
# Variability summary
var_summary = pd.DataFrame({
    "std": df[["gdp_per_capita", "life_expectancy", "gini_index"]].std(),
    "IQR": df[["gdp_per_capita", "life_expectancy", "gini_index"]].quantile(0.75) - 
           df[["gdp_per_capita", "life_expectancy", "gini_index"]].quantile(0.25),
    "CV_pct": (df[["gdp_per_capita", "life_expectancy", "gini_index"]].std() / 
               df[["gdp_per_capita", "life_expectancy", "gini_index"]].mean() * 100)
})

print(var_summary.round(2).to_string())
print()
print("The CV tells us which variable has the most RELATIVE variability:")
print(f"  GDP per capita varies the most (CV = {var_summary.loc['gdp_per_capita', 'CV_pct']:.1f}%)")
print(f"  This makes sense: income inequality across countries is enormous.")

### When SD is Misleading: Enter the IQR

Just as the mean is sensitive to outliers, so is the standard deviation. The **Interquartile Range (IQR)** — the range covering the middle 50% of data — is a robust alternative.

When SD and IQR tell different stories, you likely have outliers or heavy tails. This is a diagnostic signal.

In [ ]:
# Box plots: the best visual summary of variability
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, col, colour in zip(axes, ["gdp_per_capita", "life_expectancy", "gini_index"],
                             ["steelblue", "coral", "seagreen"]):
    box = ax.boxplot(df[col], patch_artist=True, widths=0.5,
                     boxprops=dict(facecolor=colour, alpha=0.6),
                     medianprops=dict(color="black", linewidth=2),
                     flierprops=dict(marker="o", markerfacecolor="red", markersize=5))
    ax.set_title(col.replace("_", " ").title(), fontsize=12)
    ax.set_ylabel("Value")

    # Annotate quartiles
    q1, q2, q3 = df[col].quantile([0.25, 0.5, 0.75])
    ax.annotate(f"Q1={q1:.0f}", xy=(1.15, q1), fontsize=9, color="gray")
    ax.annotate(f"Med={q2:.0f}", xy=(1.15, q2), fontsize=9, color="black", fontweight="bold")
    ax.annotate(f"Q3={q3:.0f}", xy=(1.15, q3), fontsize=9, color="gray")

plt.suptitle("Box Plots: Visualising Variability and Outliers", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("fig_00_02_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 0.5 — The Shape of Distributions: Skewness and Kurtosis

Beyond centre and spread, distributions have **shape**. Two key descriptors:

- **Skewness** measures asymmetry. Positive skewness = right tail is longer (income, house prices). Negative skewness = left tail is longer (exam scores near the maximum).
- **Kurtosis** measures the heaviness of tails. High kurtosis means more extreme values than a normal distribution would predict. In finance and econometrics, this is crucial: financial returns have *fat tails*, which is why models assuming normality underestimate risk.

### The pandas `.describe()` method — your first stop

In [ ]:
# .describe() gives you the standard five-number summary plus count, mean, and std
print(df.describe().round(2).to_string())

In [ ]:
# Skewness and kurtosis — not in .describe(), but easy to compute
shape_stats = pd.DataFrame({
    "skewness": df[["gdp_per_capita", "life_expectancy", "gini_index"]].skew(),
    "kurtosis": df[["gdp_per_capita", "life_expectancy", "gini_index"]].kurtosis()
})

print(shape_stats.round(3).to_string())
print()
print("Rules of thumb:")
print("  Skewness ≈ 0 → symmetric")
print("  Skewness > 0 → right-skewed (long right tail)")
print("  Skewness < 0 → left-skewed (long left tail)")
print()
print("  Kurtosis ≈ 0 → normal-like tails (pandas uses excess kurtosis)")
print("  Kurtosis > 0 → heavier tails than normal (leptokurtic)")
print("  Kurtosis < 0 → lighter tails than normal (platykurtic)")

---

## 0.6 — Relationships Between Variables: Correlation

In econometrics, we are almost always interested in relationships: does X affect Y? Before running regressions, we **explore** relationships visually and with correlation.

### Pearson Correlation: What It Captures and What It Misses

Pearson's *r* measures **linear** association. It ranges from -1 (perfect negative linear relationship) to +1 (perfect positive). A value near 0 means no *linear* relationship — but there could still be a strong non-linear one.

### The Correlation Matrix

In [ ]:
# Correlation matrix for numeric variables
corr_matrix = df[["gdp_per_capita", "life_expectancy", "gini_index", "population"]].corr()

# Heatmap — the standard way to visualise correlations
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Hide upper triangle (redundant)

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True, linewidths=1,
            cbar_kws={"label": "Pearson r"}, ax=ax)

ax.set_title("Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("fig_00_03_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nKey observations:")
print(f"  GDP ↔ Life expectancy: r = {corr_matrix.loc['gdp_per_capita', 'life_expectancy']:.2f}")
print(f"  GDP ↔ Gini: r = {corr_matrix.loc['gdp_per_capita', 'gini_index']:.2f}")
print(f"  Population is weakly correlated with everything — size ≠ prosperity")

### Scatter Plots: Always Visualise Before Correlating

A correlation coefficient is a single number — it can hide a multitude of patterns. **Always** plot the data first. Anscombe's quartet is the classic example, but you'll encounter this constantly in real research.

In [ ]:
# Scatter plot with regression line: GDP vs Life Expectancy
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Linear scale — relationship looks non-linear
axes[0].scatter(df["gdp_per_capita"], df["life_expectancy"], 
                alpha=0.6, color="steelblue", edgecolors="white", s=50)
axes[0].set_xlabel("GDP per Capita (USD)")
axes[0].set_ylabel("Life Expectancy (years)")
axes[0].set_title("Linear Scale")

# Log scale — relationship becomes linear
axes[1].scatter(np.log(df["gdp_per_capita"]), df["life_expectancy"],
                alpha=0.6, color="seagreen", edgecolors="white", s=50)
axes[1].set_xlabel("log(GDP per Capita)")
axes[1].set_ylabel("Life Expectancy (years)")
axes[1].set_title("Log Scale — Relationship Linearised")

# Add correlation values
r_linear = df["gdp_per_capita"].corr(df["life_expectancy"])
r_log = np.log(df["gdp_per_capita"]).corr(df["life_expectancy"])
axes[0].annotate(f"r = {r_linear:.2f}", xy=(0.05, 0.95), xycoords="axes fraction", fontsize=12, fontweight="bold")
axes[1].annotate(f"r = {r_log:.2f}", xy=(0.05, 0.95), xycoords="axes fraction", fontsize=12, fontweight="bold")

plt.suptitle("GDP and Life Expectancy: Why Transformations Matter", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig_00_04_scatter_gdp_life.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Pearson r (linear): {r_linear:.3f}")
print(f"Pearson r (log):    {r_log:.3f}")
print("\nThe log transformation reveals a STRONGER linear relationship.")
print("This is why in econometrics, log(GDP) is standard practice.")

---

## 0.7 — Group Comparisons

In research, we rarely look at the whole dataset as a monolith. We compare **groups**: countries by region, firms by sector, patients by treatment. The pandas `groupby()` method is your best friend here.

### `groupby()` — Split, Apply, Combine

In [ ]:
# Summary statistics by region
regional = df.groupby("region")[["gdp_per_capita", "life_expectancy", "gini_index"]].agg(
    ["count", "mean", "median", "std"]
)

# Flatten multi-level columns for readability
regional.columns = [f"{var}_{stat}" for var, stat in regional.columns]

print("Summary by Region:")
print(regional.round(1).to_string())

In [ ]:
# Visual comparison: violin plots by region
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.violinplot(data=df, x="region", y="gdp_per_capita", hue="region",
               palette="Set2", inner="quartile", cut=0, legend=False, ax=axes[0])
axes[0].set_title("GDP per Capita by Region")
axes[0].set_ylabel("USD")
axes[0].tick_params(axis="x", rotation=30)

sns.violinplot(data=df, x="region", y="life_expectancy", hue="region",
               palette="Set2", inner="quartile", cut=0, legend=False, ax=axes[1])
axes[1].set_title("Life Expectancy by Region")
axes[1].set_ylabel("Years")
axes[1].tick_params(axis="x", rotation=30)

plt.suptitle("Regional Comparisons: Violin Plots", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("fig_00_05_violin_by_region.png", dpi=150, bbox_inches="tight")
plt.show()

print("Violin plots show the FULL distribution, not just summary statistics.")
print("Much richer than bar charts with error bars — always prefer them.")

---

## 0.8 — Pandas Essentials Cheatsheet

Here's a quick reference for the pandas operations we used. You'll use these constantly.

| Operation | Syntax | What it does |
|-----------|--------|-------------|
| Load CSV | `pd.read_csv("file.csv")` | Read tabular data into a DataFrame |
| Inspect | `df.head()`, `df.shape`, `df.dtypes`, `df.info()` | First look at your data |
| Summary stats | `df.describe()` | Count, mean, std, min, quartiles, max |
| Single stat | `df["col"].mean()`, `.median()`, `.std()` | Compute one statistic |
| Multiple stats | `df["col"].agg(["mean", "median", "std"])` | Compute several at once |
| Skewness/Kurtosis | `df["col"].skew()`, `df["col"].kurtosis()` | Distribution shape |
| Correlation | `df.corr()` | Pairwise Pearson correlations |
| Group by | `df.groupby("col").agg(...)` | Split-apply-combine |
| Filter rows | `df[df["col"] > value]` | Boolean indexing |
| New column | `df["new"] = expression` | Vectorised column creation |
| Log transform | `np.log(df["col"])` | Element-wise natural log |

---

## ✏️ Exercises

Now it's your turn. These exercises reinforce both the statistical concepts and the pandas skills from this module.

### Exercise 0.1 — Exploring a Subset

Filter the dataset to only **African countries**. Compute the mean, median, and standard deviation of GDP per capita for this subset. How does the mean–median gap compare to the full dataset? What does this tell you about income distribution within Africa?

In [ ]:
# YOUR CODE HERE
# Hint: df[df["region"] == "Africa"]


### Exercise 0.2 — Creating a New Variable

Create a new column called `gdp_log` containing the natural log of GDP per capita. Then compute the skewness of both `gdp_per_capita` and `gdp_log`. Which is closer to zero? Why does this matter for regression?

In [ ]:
# YOUR CODE HERE


### Exercise 0.3 — The Outlier Question

A single country has GDP per capita of $500,000 (an extreme outlier — think of a tax haven). Add this observation to the dataset and recompute the mean and median of GDP per capita. Which measure changes more? By how much?

This exercise illustrates why **robustness** matters in descriptive statistics.

In [ ]:
# YOUR CODE HERE
# Hint: use pd.concat() to add a new row


### Exercise 0.4 — Grouped Visualisation

Create a box plot of `gini_index` grouped by `region`. Which region has the highest inequality? Which has the most variability in inequality?

In [ ]:
# YOUR CODE HERE


### Exercise 0.5 — Correlation ≠ Causation (Thinking Exercise)

In our dataset, GDP per capita and life expectancy are positively correlated. List at least **three** reasons why we cannot conclude that higher GDP *causes* longer life. For each reason, identify whether it's a confounding variable, reverse causation, or selection bias.

*Write your answer as a markdown cell below or as comments in the code cell.*

In [ ]:
# YOUR REASONING HERE (as comments)
#
# Reason 1: ...
# Type: confounding / reverse causation / selection bias
#
# Reason 2: ...
# Type: ...
#
# Reason 3: ...
# Type: ...


---

## 📌 Key Takeaways

1. **Always inspect your data first** — `.shape`, `.dtypes`, `.describe()`, `.head()`
2. **Mean vs. median gap** is a diagnostic for skewness — when they diverge, investigate
3. **Log transformations** tame right-skewed variables and are standard in econometrics
4. **Coefficient of Variation** lets you compare variability across different scales
5. **Visualise before computing** — a correlation coefficient hides patterns that scatter plots reveal
6. **`groupby()`** is the workhorse for comparing groups in pandas
7. **Correlation ≠ causation** — always think about confounders, reverse causation, and selection

---

## 🔜 Next Module

**Module 01 — Probability Foundations**: Random experiments, events, axioms of probability, conditional probability, Bayes' theorem. We build the language needed for all of inferential statistics.

---

*This notebook is part of the open course "Intermediate Statistics for Econometrics" by Andrea Sarcina.*  
*Published at [cinosar.github.io](https://cinosar.github.io) — Licensed under CC BY 4.0*